# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-azeemi/fatima-flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
## 1. Question

The research question is:

**Which safe, publicly observable signals are associated with a content page entering decline, and can these signals be used to prioritize pages for editorial review?**

The decision this work supports is **editorial prioritization**: deciding which pages a human editor should review first.

The analysis uses six pre-outcome features:

- `word_count`
- `content_age_days`
- `impressions_90d`
- `avg_position`
- `ctr`
- `search_volume`

The target is `is_declining`, derived from `trend_direction == "down"`.

The main hypothesis tested was whether older, high-traffic pages are more likely to decline. The results did not support that assumption. In this snapshot, younger and lower-volume pages showed higher observed decline rates.

The model is therefore treated as a **decision-support system**, not as an automated content-editing or ranking system.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
## 2. Data

The analysis uses the anonymized `content_refresh_anonymized.csv` release from the FlyRank ML Internship starter repository.

The dataset contains **30,000 content pages**. Each row represents one content page identified by `content_id`.

The observation window is centered on **March 2026**.

The dataset contains:

- 30,000 total pages
- 16,262 declining pages
- approximately 54.2% decline base rate
- 6 model features

### Features used

The model uses only signals available before the prediction point:

- `word_count`
- `content_age_days`
- `impressions_90d`
- `avg_position`
- `ctr`
- `search_volume`

### Target

The binary target `is_declining` is created as:

`trend_direction == "down"`

Therefore, pages with a `trend_direction` of `down` are treated as the positive class.

### Excluded fields

`trend_pct` was excluded because it directly represents percentage change in the outcome and would introduce severe label leakage.

`trend_direction` was excluded from the feature matrix because it is the source of the target label.

`content_id` and `client_id` were retained only for identification and joining purposes and were never provided to the model.

Short-window fields overlapping with the 90-day aggregates were also excluded to reduce leakage and duplicated information.

### Missing values

`word_count` has approximately 25.7% missing values and `search_volume` approximately 8.2%.

Missing values are handled using **training-set median imputation only**. The imputer is fitted on the training data and then applied to the held-out test data.

This prevents information from the test set from influencing model training.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
## 3. Methodology

This task is framed as binary risk scoring.

The model estimates the probability that a page belongs to the declining class:

`trend_direction == "down"`

### Baseline

The baseline is the existing Week-4 rule:

`STALE_HIGH_IMPRESSIONS`

A page is flagged when:

- `content_age_days >= 90`
- `impressions_90d >= 500`

This represents the common assumption that older, high-visibility pages are more likely to decline.

### Train/test split

The dataset is divided using an **80/20 stratified train/test split** with:

`random_state = 42`

Stratification preserves approximately the same 54.2% decline rate in both folds.

The split is performed before imputation.

### Leakage control

Potential leakage was checked by examining correlations between candidate features and the target.

A Pearson correlation threshold of 0.85 was used as a warning threshold.

Label-derived variables such as `trend_direction` and `trend_pct` were excluded from the feature matrix.

Identifiers such as `content_id` and `client_id` were also excluded.

### Model

The final model is a Random Forest Classifier with:

- `n_estimators = 150`
- `max_depth = 8`
- `min_samples_split = 10`
- `random_state = 42`

Random Forest was selected because the six metadata and search signals can interact non-linearly and may contain skewed distributions.

### Evaluation metrics

The model is evaluated using:

- Accuracy
- Precision
- Recall
- F1 score
- ROC-AUC

The main validation result is reported on the unseen test set.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
## 4. Results (vs baseline)

The Random Forest was evaluated on the same held-out test set used for the baseline comparison.

The results show that the Random Forest performs substantially better than the existing stale-content rule.

| Strategy | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Always predict "down" | 0.542 | 0.542 | 1.000 | 0.703 | — |
| STALE_HIGH_IMPRESSIONS rule | 0.482 | 0.536 | 0.328 | 0.407 | — |
| **Random Forest** | **0.677** | **0.665** | **0.815** | **0.732** | **0.737** |

The Random Forest achieved a test accuracy of **67.7%**, F1 score of **0.732**, and ROC-AUC of **0.737**.

The rule baseline achieved only **48.2% accuracy** and **0.407 F1**, performing worse than the simple always-declining base-rate strategy on these metrics.

The train/test comparison was:

| Metric | Training | Test |
|---|---:|---:|
| Accuracy | 0.700 | 0.677 |
| Precision | 0.679 | 0.665 |
| Recall | 0.845 | 0.815 |
| F1 | 0.753 | 0.732 |
| ROC-AUC | 0.772 | 0.737 |

The relatively small gap between training and test performance suggests mild generalization loss rather than severe overfitting.

### Feature importance

The Random Forest feature importance ranking was:

1. `impressions_90d` — 34.3%
2. `avg_position` — 23.3%
3. `content_age_days` — 20.9%
4. `word_count` — 9.27%
5. `ctr` — 7.15%
6. `search_volume` — 5.03%

The three strongest features — impressions, average position, and content age — account for approximately 79% of the model's feature importance.

### Main finding

The strongest finding is a reversal of the original editorial assumption.

Observed decline rates were:

| Content age | Decline rate |
|---|---:|
| <90 days | 66.9% |
| 90–180 days | 62.6% |
| 181–365 days | 51.5% |
| >365 days | 42.6% |

Search-volume tiers showed a similar pattern:

| Search volume | Decline rate |
|---|---:|
| Zero/Low | 59.0% |
| Medium | 52.1% |
| High | 50.1% |
| Very High | 44.5% |

Therefore, in this snapshot, younger and lower-volume pages showed higher observed decline rates.

This does **not** mean that old pages are safe or that young pages should automatically be rewritten. Age and traffic are better interpreted as separate **risk and impact signals** rather than as proof of causation.

## 5. Limitations

*What this work cannot claim.*

In [ ]:
## 5. Limitations

This analysis has several important limitations.

### 1. Observational data

The results show associations, not causal relationships.

The analysis cannot demonstrate that updating content, adding words, changing dates, or refreshing a page will cause traffic to recover.

### 2. Not a Google ranking model

The model does not predict Google's ranking algorithm, core updates, manual actions, or future algorithm changes.

It predicts an internal decline label derived from the available historical snapshot.

### 3. Cross-sectional snapshot

The dataset is a snapshot rather than a longitudinal panel of the same pages over multiple periods.

Therefore, the observed age-related pattern may partly reflect cohort, seasonal, or topical effects.

### 4. Limited feature set

Only six safe metadata/search features were used.

A test ROC-AUC of 0.737 shows useful predictive signal, but it also indicates that substantial variation in page-level decline is not explained by these features.

### 5. Young pages have incomplete history

Pages younger than 90 days may not have a complete historical observation window.

Their metrics can therefore be noisier than those of older pages.

### 6. Correlated time windows

The source data contains overlapping 30-day and 90-day measurement windows.

These should therefore be interpreted as correlated moving averages rather than independent observations.

### 7. Human review remains necessary

The model should be used to prioritize editorial review, not to automatically rewrite, redirect, deprecate, or delete content.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
## 6. Ranked Recommendations

The model produces a ranked action queue based on predicted decline probability.

The final action playbook contains three recommendation groups:

| Action | Pages |
|---|---:|
| REFRESH_CONTENT | 16,726 |
| REVIEW_STALE | 8,057 |
| MONITOR | 5,217 |

### 1. Refresh content — STALE_HIGH_IMPRESSIONS

Prioritize pages with:

- age >= 90 days
- impressions >= 500
- elevated model risk probability

These pages are prioritized because they have significant traffic exposure to protect.

Importantly, they are **not** necessarily the pages most likely to decline.

The confidence is therefore high for the impact/prioritization dimension but only moderate for individual-page risk.

Human verification is required before making content changes.

### 2. Review stale, low-traffic pages — STALE_LOW_IMPRESSIONS

Pages older than 180 days with low impressions should be reviewed for:

- keyword misalignment
- weak search demand
- consolidation opportunities
- limited traffic potential

A simple content refresh may not be appropriate if there was little traffic to protect.

### 3. Monitor — LOW_PRIORITY

Newer or stable pages should generally be monitored rather than immediately rewritten.

This is especially important for pages younger than 90 days because the analysis observed higher volatility in this group.

### Required human checks

Before acting on a flagged page, an editor should verify:

1. Whether the page serves evergreen or time-sensitive intent.
2. Whether the traffic change could be seasonal or campaign-related.
3. Whether the page is already redirected or scheduled for deprecation.

### No-go actions

The model should never independently trigger:

- mass AI rewriting
- automatic publishing
- canonical redirection
- content deletion
- page deprecation

All such actions require human editorial approval.

### Monitoring

The model should be retrained on a rolling 90-day window at least quarterly.

An earlier retraining cycle should be considered if the feature distribution shifts substantially or if Precision@50 on newly flagged pages falls meaningfully below the validation baseline.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
## 7. Artifacts the Paper Embeds

The analysis produces the following artifacts for the deployed research paper:

1. Trend-direction distribution
2. Model vs baseline performance comparison
3. Train vs test performance comparison
4. Random Forest feature importance
5. Decline rate by content-age tier
6. Decline rate by search-volume tier
7. Recommended action distribution
8. Reason-code distribution
9. Ranked action queue
10. Model results summary

These artifacts provide both model-performance evidence and an interpretable view of how the model supports editorial prioritization.

In [1]:
import pandas as pd

results_table = pd.DataFrame({
    "Strategy": [
        "Always predict down",
        "STALE_HIGH_IMPRESSIONS",
        "Random Forest"
    ],
    "Accuracy": [0.542, 0.482, 0.677],
    "Precision": [0.542, 0.536, 0.665],
    "Recall": [1.000, 0.328, 0.815],
    "F1": [0.703, 0.407, 0.732],
    "ROC_AUC": [None, None, 0.737]
})

results_table

,Strategy,Accuracy,Precision,Recall,F1,ROC_AUC
0,Always predict down,0.542,0.542,1.000,0.703,NaN
1,STALE_HIGH_IMPRESSIONS,0.482,0.536,0.328,0.407,NaN
2,Random Forest,0.677,0.665,0.815,0.732,0.737


In [2]:
import pandas as pd

results_table = pd.DataFrame({
    "Strategy": [
        "Always predict down",
        "STALE_HIGH_IMPRESSIONS",
        "Random Forest"
    ],
    "Accuracy": [0.542, 0.482, 0.677],
    "Precision": [0.542, 0.536, 0.665],
    "Recall": [1.000, 0.328, 0.815],
    "F1": [0.703, 0.407, 0.732],
    "ROC_AUC": [None, None, 0.737]
})

results_table

,Strategy,Accuracy,Precision,Recall,F1,ROC_AUC
0,Always predict down,0.542,0.542,1.000,0.703,NaN
1,STALE_HIGH_IMPRESSIONS,0.482,0.536,0.328,0.407,NaN
2,Random Forest,0.677,0.665,0.815,0.732,0.737


In [4]:
feature_importance = pd.DataFrame({
    "Feature": [
        "impressions_90d",
        "avg_position",
        "content_age_days",
        "word_count",
        "ctr",
        "search_volume"
    ],
    "Importance": [
        34.3,
        23.3,
        20.9,
        9.27,
        7.15,
        5.03
    ]
})

feature_importance

,Feature,Importance
0,impressions_90d,34.30
1,avg_position,23.30
2,content_age_days,20.90
3,word_count,9.27
4,ctr,7.15
5,search_volume,5.03


In [6]:
action_summary = pd.DataFrame({
    "Action": [
        "REFRESH_CONTENT",
        "REVIEW_STALE",
        "MONITOR"
    ],
    "Pages": [
        16726,
        8057,
        5217
    ]
})

action_summary

,Action,Pages
0,REFRESH_CONTENT,16726
1,REVIEW_STALE,8057
2,MONITOR,5217


In [7]:
action_summary = pd.DataFrame({
    "Action": [
        "REFRESH_CONTENT",
        "REVIEW_STALE",
        "MONITOR"
    ],
    "Pages": [
        16726,
        8057,
        5217
    ]
})

action_summary

,Action,Pages
0,REFRESH_CONTENT,16726
1,REVIEW_STALE,8057
2,MONITOR,5217


In [8]:
search_decline = pd.DataFrame({
    "Search_Volume": [
        "Zero/Low",
        "Medium",
        "High",
        "Very High"
    ],
    "Decline_Rate": [
        59.0,
        52.1,
        50.1,
        44.5
    ]
})

search_decline

,Search_Volume,Decline_Rate
0,Zero/Low,59.0
1,Medium,52.1
2,High,50.1
3,Very High,44.5


### 8. 5-Minute Demo Outline

In [ ]:
# ML-12 — 5-Minute Demo Outline

## 0:00–0:45 — Problem

Content teams manage thousands of pages but have limited editorial capacity.

The goal of this project is to identify which pages should be reviewed first when there are signs of potential content decline.

The system is designed as decision-support rather than an automated editing system.

## 0:45–1:30 — Data

The project uses an anonymized 30,000-page snapshot.

Six safe features are used:

- word count
- content age
- 90-day impressions
- average position
- CTR
- search volume

The target is whether `trend_direction == "down"`.

## 1:30–2:30 — Methodology

The dataset is split using an 80/20 stratified split with seed 42.

Missing values are imputed using training-set medians only.

A stale-content rule is used as the baseline.

The final model is a Random Forest with 150 trees and controlled depth.

Leakage checks exclude outcome-derived fields and identifiers.

## 2:30–3:30 — Results

The baseline rule achieves:

- 48.2% accuracy
- 0.407 F1

The Random Forest achieves:

- 67.7% accuracy
- 0.732 F1
- 0.737 ROC-AUC

The train/test gap remains relatively small.

## 3:30–4:15 — Main Finding

The major finding is that older and higher-volume pages were not the most likely to decline.

Observed decline rates were higher among younger and lower-volume pages.

This challenges the assumption that old, high-traffic pages automatically represent the highest decline risk.

## 4:15–5:00 — Action Queue

The model produces a ranked editorial queue:

- 16,726 pages — REFRESH_CONTENT
- 8,057 pages — REVIEW_STALE
- 5,217 pages — MONITOR

The queue is not an automated publishing system.

Editors must review flagged pages before taking action.

### 9. Employer-Facing Summary

In [ ]:
# ML-12 — Employer-Facing Summary

I built a leakage-checked machine learning pipeline to identify content pages associated with search-performance decline using an anonymized 30,000-page dataset. Using six safe metadata and search signals, a Random Forest achieved **0.737 test ROC-AUC, 0.732 F1, and 67.7% accuracy**, outperforming the existing stale-content rule.

The analysis also challenged a common editorial assumption: older and higher-traffic pages were not the most likely to decline in this snapshot. I translated the model output into a ranked editorial action queue while keeping human review in the loop and explicitly avoiding automated rewriting, deletion, redirection, or publishing decisions.

This project demonstrates practical skills in **data preparation, leakage prevention, supervised machine learning, model evaluation, feature interpretation, reproducible experimentation, and translating ML predictions into actionable decision-support workflows**.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
